# Milestone 4 (Revised) - SHAP, Feature Selection, Nested Tuning
**DATA4402 Data Science Capstone - Austin Novak**

This notebook answers the instructor's four requests on the Module 4 submission:

| # | Request | Where |
|---|---------|-------|
| 1 | Explain the existing models with **SHAP** (which of the original seven features matter most, per model) | Section 4 |
| 2 | Evaluate additional features with **RFE and permutation importance**, compare with the original seven, justify each addition, keep selection inside the training data of each validation split | Sections 5-6 |
| 3 | Combine **GridSearchCV with five-fold CV** (grouped by `SITE_ID`) for the existing models plus a few new ones | Section 3 and 7 |
| 4 | Compare the original and expanded feature sets **fairly**, report hyperparameters and Macro-F1 / Accuracy / Top-3, keep held-out test sites out of selection and tuning, separate grid-search scores from test results, discuss honestly | Sections 7-9 |

**Protocol (nested, grouped).** Held-out **test sites** are set aside first and are touched once, at the very end. The remaining
*development* sites go through an outer StratifiedGroupKFold. Inside every outer fold, using **only that fold's training sites**:
(a) feature selection (RFE + permutation importance with random-noise probes), then (b) GridSearchCV with an inner 5-fold
StratifiedGroupKFold. The outer validation sites are used only to score. Three arms are compared on identical folds with identical grids:

* **A** - original 7 features, fixed settings (the Module 4 style default, no tuning)
* **B** - original 7 features, tuned
* **C** - original 7 + selected additions, tuned (same grids as B)

In [1]:
import os, json, time, gc, pickle, warnings, itertools
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, f1_score, log_loss, roc_auc_score, confusion_matrix, classification_report)
import m4_utils
warnings.filterwarnings("ignore")

FAST = False                     # True = tiny smoke test
RND = 42
N_OUTER = 5 if not FAST else 2   # outer validation folds on the development sites
N_JOBS = 2
N_SHAP, N_BG = (150, 30) if not FAST else (20, 10)
FIG, RES, MOD, CACHE = "figures", "results", "models", "cache"
for d in (FIG, RES, MOD, CACHE): os.makedirs(d, exist_ok=True)
RESULTS = {}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold", "font.size": 10, "legend.frameon": False})
BLUE, ORANGE, GREEN, RED, GREY = "#4C72B0", "#DD8452", "#55A868", "#C44E52", "#9AA0A6"
def savefig(n): plt.savefig(f"{FIG}/{n}.png", dpi=150, bbox_inches="tight")
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def cached(name, fn):
    """Run fn() once and keep the result on disk, so an interrupted run can resume."""
    p = f"{CACHE}/{name}{'_fast' if FAST else ''}.pkl"
    if os.path.exists(p):
        with open(p, "rb") as fh: return pickle.load(fh)
    out = fn()
    with open(p, "wb") as fh: pickle.dump(out, fh)
    return out

## 1. Data and candidate features
The cleaned Module 3/4 table (`cleaned_dataset.csv`) is joined to extra columns of the original spreadsheet
(key = `SITE_ID` + `Station_Date` + `species`; the join is checked to be exact). The seven **original** features are the Module 4 inputs
`LAT_DD, LON_DD, month, lcond, PHSTVL, state, ECOREGL3`. The **candidate pool** adds 20 more columns. Three were dropped *before* any modeling for
logical reasons: `WS_AREA` (same quantity as `LWSKM2` before log), `hco3` (twin of `alk`), `BigHUC` (a copy of `HUC`).
Columns with >90% missing (`FE, MN, al, Se, ...`) and fish-community summary columns (`ABUND, NUMSPEC, ...`, which describe the fish and would leak the answer) are excluded.

In [2]:
clean = pd.read_csv("cleaned_dataset.csv"); clean["SITE_ID"] = clean["SITE_ID"].astype(str)
raw = pd.read_excel("data/Combined_Less_DNR Fish Data - Final - Austin Novak.xlsx"); raw["SITE_ID"] = raw["SITE_ID"].astype(str)
ORIG_NUM = ["LAT_DD", "LON_DD", "month", "lcond", "PHSTVL"]; ORIG_CAT = ["state", "ECOREGL3"]
ORIG7 = ORIG_NUM + ORIG_CAT
ADD_NUM = ["YEAR", "Temp", "LWSKM2", "CA", "SO4", "NO3", "CL", "alk", "mg", "na", "k", "tn", "hardness", "SO4HCO3", "ratio", "rbpscore", "do", "TP"]
ADD_CAT = ["Dataset", "HUC"]
key = ["SITE_ID", "Station_Date", "species"]
extra = raw.drop_duplicates(key)[key + [c for c in ADD_NUM + ADD_CAT if c not in clean.columns]]
data = clean.merge(extra, on=key, how="left", validate="m:1")
assert len(data) == len(clean) == 42144 and data["Dataset"].notna().all()
data["ECOREGL3"] = data["ECOREGL3"].astype(str); data["HUC"] = data["HUC"].astype(str)
CANDIDATES = ORIG7 + ADD_NUM + ADD_CAT
CAT_COLS = ORIG_CAT + ADD_CAT
GEO_NUM = ["LAT_DD", "LON_DD", "month", "YEAR"]
CHEM_NUM = [c for c in CANDIDATES if c not in CAT_COLS + GEO_NUM]
if FAST:
    ks = np.random.RandomState(RND).choice(data["SITE_ID"].unique(), int(0.25 * data["SITE_ID"].nunique()), replace=False)
    data = data[data["SITE_ID"].isin(ks)].reset_index(drop=True)
le = LabelEncoder(); y = le.fit_transform(data["GENUS"]); K = len(le.classes_); groups = data["SITE_ID"].values
X = data[CANDIDATES].copy()
print(f"{len(data):,} rows | {data.SITE_ID.nunique():,} sites | {K} genera | {len(CANDIDATES)} candidate columns ({len(ORIG7)} original + {len(CANDIDATES)-len(ORIG7)} extra)")
miss = pd.DataFrame({"pct_missing": (X.isna().mean() * 100).round(1), "original7": [c in ORIG7 for c in X.columns]}).sort_values("pct_missing", ascending=False)
display(miss.T)
miss.to_csv(f"{RES}/candidate_feature_missingness.csv")
RESULTS["dataset"] = dict(rows=len(data), sites=int(data.SITE_ID.nunique()), genera=K, n_candidates=len(CANDIDATES))

42,144 rows | 3,278 sites | 50 genera | 27 candidate columns (7 original + 20 extra)


,TP,do,hardness,rbpscore,k,na,mg,tn,ratio,alk,...,LON_DD,month,YEAR,PHSTVL,state,ECOREGL3,lcond,LAT_DD,Dataset,HUC
pct_missing,82.7,73.5,68.6,63.7,56.3,56.2,55.3,53.4,53.1,53.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
original7,False,False,False,False,False,False,False,False,False,False,...,True,True,False,True,True,True,True,True,False,False


## 2. Splits: held-out test sites, then outer folds on the development sites
Every split is by **stream site** (all rows from one site stay together), stratified by genus. Test sites are never used for feature
selection, tuning, SHAP-based decisions or model choice.

In [3]:
sg = StratifiedGroupKFold(5, shuffle=True, random_state=RND)
folds0 = [te for _, te in sg.split(X, y, groups)]
test_idx = folds0[0]; dev_idx = np.setdiff1d(np.arange(len(data)), test_idx)
Xdev, ydev, gdev = X.iloc[dev_idx].reset_index(drop=True), y[dev_idx], groups[dev_idx]
Xte, yte, gte = X.iloc[test_idx].reset_index(drop=True), y[test_idx], groups[test_idx]
assert not set(gdev) & set(gte)
outer_cv = StratifiedGroupKFold(5, shuffle=True, random_state=RND + 1)
OUTER = [(tr, va) for tr, va in outer_cv.split(Xdev, ydev, gdev)][:N_OUTER]
for tr, va in OUTER: assert not set(gdev[tr]) & set(gdev[va])
tbl = pd.DataFrame([dict(split="held-out TEST", rows=len(Xte), sites=len(set(gte)))] + [dict(split="development (total)", rows=len(Xdev), sites=len(set(gdev)))] +
    [dict(split=f"outer fold {i+1}: train / validation", rows=f"{len(tr):,} / {len(va):,}", sites=f"{len(set(gdev[tr])):,} / {len(set(gdev[va])):,}") for i, (tr, va) in enumerate(OUTER)])
display(tbl); tbl.to_csv(f"{RES}/split_summary.csv", index=False)
cnt = pd.Series(y).value_counts(); RESULTS["imbalance_ratio"] = float(cnt.max() / cnt.min())
print(f"Class imbalance: {cnt.max():,} rows (most common genus) vs {cnt.min()} (rarest) = {cnt.max()/cnt.min():.0f}:1  -> class weights are used (Module 4 result)")

,split,rows,sites
0,held-out TEST,8428,656
1,development (total),33716,2622
2,outer fold 1: train / validation,"26,974 / 6,742","2,097 / 525"
3,outer fold 2: train / validation,"26,975 / 6,741","2,098 / 524"
4,outer fold 3: train / validation,"26,976 / 6,740","2,093 / 529"
5,outer fold 4: train / validation,"26,975 / 6,741","2,096 / 526"
6,outer fold 5: train / validation,"26,964 / 6,752","2,104 / 518"


Class imbalance: 5,254 rows (most common genus) vs 33 (rarest) = 159:1  -> class weights are used (Module 4 result)


## 3. Shared preprocessing, scoring, models and grids
One preprocessing recipe for every model and every arm, fit inside each training fold only: IQR winsorizing of chemistry columns (outliers capped, never deleted),
median imputation **with missing-flags**, standard scaling, one-hot encoding (rare categories pooled). `build_pre(features)` builds it for any feature list.
Metrics: **Macro-F1** (primary; GridSearchCV refits on it), **Accuracy**, **Top-3** (true genus in the 3 most probable).

In [4]:
def build_pre(feats):
    geo = [c for c in feats if c in GEO_NUM]; chem = [c for c in feats if c in CHEM_NUM]; cat = [c for c in feats if c in CAT_COLS]
    parts = []
    if geo: parts.append(("geo", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), geo))
    if chem: parts.append(("chem", Pipeline([("w", m4_utils.IQRWinsorizer(k=1.5)), ("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), chem))
    if cat: parts.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30, sparse_output=False), cat))
    return ColumnTransformer(parts)

def full_proba(model, Xq):
    P = model.predict_proba(Xq); out = np.zeros((len(Xq), K)); out[:, model.classes_] = P; return out
def top3_of(full, yq): return float((np.argsort(full, 1)[:, -3:] == np.asarray(yq)[:, None]).any(1).mean())
def top3_scorer(est, Xq, yq): return top3_of(full_proba(est, Xq), yq)
def evaluate(model, Xq, yq, extra=False):
    full = full_proba(model, Xq); pred = full.argmax(1)
    d = dict(macro_f1=f1_score(yq, pred, average="macro", zero_division=0), accuracy=accuracy_score(yq, pred), top3=top3_of(full, yq))
    if extra:
        aucs = [roc_auc_score(yq == c, full[:, c]) for c in np.unique(yq) if 0 < (yq == c).sum() < len(yq)]
        d.update(roc_auc=float(np.mean(aucs)), log_loss=log_loss(yq, np.clip(full, 1e-6, 1), labels=np.arange(K)))
    return d, pred, full

SCORING = {"macro_f1": "f1_macro", "accuracy": "accuracy", "top3": top3_scorer}
SPECS = {
 "Logistic Regression": dict(est=LogisticRegression(max_iter=300, random_state=RND), fixed=dict(class_weight="balanced"),
        grid={"model__C": [0.1, 1.0], "model__class_weight": [None, "balanced"]}, sw=False, existing=True),
 "Random Forest": dict(est=RandomForestClassifier(n_estimators=100, random_state=RND, n_jobs=1), fixed=dict(max_depth=20, min_samples_leaf=5, class_weight="balanced_subsample"),
        grid={"model__max_depth": [10, 20], "model__min_samples_leaf": [5, 20], "model__class_weight": ["balanced_subsample"]}, sw=False, existing=True),
 "Gradient Boosting": dict(est=HistGradientBoostingClassifier(max_iter=25, random_state=RND), fixed=dict(learning_rate=0.1, max_depth=3, class_weight="balanced"),
        grid={"model__learning_rate": [0.05, 0.15], "model__max_depth": [3, 5], "model__class_weight": ["balanced"]}, sw=False, existing=True),
 "Neural Network": dict(est=MLPClassifier(max_iter=30, early_stopping=True, n_iter_no_change=4, random_state=RND), fixed=dict(hidden_layer_sizes=(64,), alpha=1e-4),
        grid={"model__hidden_layer_sizes": [(64,), (64, 32)], "model__alpha": [1e-4, 1e-2]}, sw=True, existing=True),
 "Extra Trees (new)": dict(est=ExtraTreesClassifier(n_estimators=100, random_state=RND, n_jobs=1), fixed=dict(max_depth=20, min_samples_leaf=5, class_weight="balanced_subsample"),
        grid={"model__max_depth": [10, 20], "model__min_samples_leaf": [5, 20], "model__class_weight": ["balanced_subsample"]}, sw=False, existing=False),
 "kNN (new)": dict(est=KNeighborsClassifier(n_jobs=1), fixed=dict(n_neighbors=25, weights="distance"),
        grid={"model__n_neighbors": [15, 50], "model__weights": ["uniform", "distance"]}, sw=False, existing=False),
}
if FAST:
    for s in SPECS.values(): s["grid"] = {k: v[:1] if i else v[:2] for i, (k, v) in enumerate(s["grid"].items())}
rows = []
for n, s in SPECS.items():
    nc = int(np.prod([len(v) for v in s["grid"].values()]))
    for p, v in s["grid"].items(): rows.append(dict(model=n, existing_model=s["existing"], parameter=p.replace("model__", ""), values=str(v), combinations=nc, inner_cv_fits=nc * 5))
    rows.append(dict(model=n, existing_model=s["existing"], parameter="(arm A fixed settings)", values=str(s["fixed"]), combinations=1, inner_cv_fits=0))
grid_tbl = pd.DataFrame(rows); display(grid_tbl); grid_tbl.to_csv(f"{RES}/parameter_grids.csv", index=False)
print("Total grid combinations per tuned arm:", sum(int(np.prod([len(v) for v in s['grid'].values()])) for s in SPECS.values()), "- identical for arm B and arm C (same budget).")

def make_pipe(name, feats, fixed=False):
    est = clone(SPECS[name]["est"])
    if fixed: est.set_params(**SPECS[name]["fixed"])
    return Pipeline([("pre", build_pre(feats)), ("model", est)])

def fit_fixed(name, feats, Xtr, ytr):
    p = make_pipe(name, feats, fixed=True)
    kw = {"model__sample_weight": compute_sample_weight("balanced", ytr)} if SPECS[name]["sw"] else {}
    return p.fit(Xtr[feats], ytr, **kw)

def tune(name, feats, Xtr, ytr, gtr):
    """GridSearchCV: inner 5-fold StratifiedGroupKFold grouped by SITE_ID, refit on macro-F1."""
    s = SPECS[name]
    gs = GridSearchCV(make_pipe(name, feats), s["grid"], cv=StratifiedGroupKFold(5, shuffle=True, random_state=RND), scoring=SCORING,
                      refit="macro_f1", n_jobs=N_JOBS, error_score="raise")
    kw = {"model__sample_weight": compute_sample_weight("balanced", ytr)} if s["sw"] else {}
    gs.fit(Xtr[feats], ytr, groups=gtr, **kw)
    return gs

def best_params(gs): return {k.replace("model__", ""): (list(v) if isinstance(v, tuple) else v) for k, v in gs.best_params_.items()}

# quick timing so the run budget is visible
t0 = time.time(); _ = fit_fixed("Random Forest", ORIG7, Xdev, ydev); print(f"Reference: one Random Forest fit on the development set takes {time.time()-t0:.1f}s"); del _

,model,existing_model,parameter,values,combinations,inner_cv_fits
0,Logistic Regression,True,C,"[0.1, 1.0]",4,20
1,Logistic Regression,True,class_weight,"[None, 'balanced']",4,20
2,Logistic Regression,True,(arm A fixed settings),{'class_weight': 'balanced'},1,0
3,Random Forest,True,max_depth,"[10, 20]",4,20
4,Random Forest,True,min_samples_leaf,"[5, 20]",4,20
5,Random Forest,True,class_weight,['balanced_subsample'],4,20
6,Random Forest,True,(arm A fixed settings),"{'max_depth': 20, 'min_samples_leaf': 5, 'clas...",1,0
7,Gradient Boosting,True,learning_rate,"[0.05, 0.15]",4,20
8,Gradient Boosting,True,max_depth,"[3, 5]",4,20
9,Gradient Boosting,True,class_weight,['balanced'],4,20


Total grid combinations per tuned arm: 24 - identical for arm B and arm C (same budget).


Reference: one Random Forest fit on the development set takes 2.9s


## 4. SHAP: which of the original seven features drive each existing model?
The four existing models (Logistic Regression, Random Forest, Gradient Boosting, Neural Network) are tuned on the **development** sites with the original 7 features
(GridSearchCV, five-fold grouped), then explained with **Kernel SHAP** computed directly on the 7 raw input columns. With 7 features the Shapley
values are exact (all 128 feature subsets are evaluated); "removed" features are replaced by a background sample of development rows. SHAP values are in probability units
and are explained on a random sample of **development** sites (test sites are not used). Importance = mean |SHAP| summed over the genus outputs, shown as a share of the total.

In [5]:
def tune_B_final():
    out = {}
    for n in SPECS:
        t0 = time.time(); gs = tune(n, ORIG7, Xdev, ydev, gdev)
        out[n] = dict(est=gs.best_estimator_, params=best_params(gs), cv=pd.DataFrame(gs.cv_results_), best_cv=gs.best_score_, secs=time.time() - t0)
        print(f"  {n:22s} best inner-CV macro-F1 {gs.best_score_:.4f}  {best_params(gs)}  ({out[n]['secs']/60:.1f} min)", flush=True)
    return out
FINAL_B = cached("final_B", tune_B_final)

  Logistic Regression    best inner-CV macro-F1 0.0252  {'C': 1.0, 'class_weight': 'balanced'}  (0.6 min)


  Random Forest          best inner-CV macro-F1 0.0467  {'class_weight': 'balanced_subsample', 'max_depth': 20, 'min_samples_leaf': 5}  (0.4 min)


  Gradient Boosting      best inner-CV macro-F1 0.0408  {'class_weight': 'balanced', 'learning_rate': 0.15, 'max_depth': 5}  (1.0 min)


  Neural Network         best inner-CV macro-F1 0.0280  {'alpha': 0.01, 'hidden_layer_sizes': [64, 32]}  (0.4 min)


  Extra Trees (new)      best inner-CV macro-F1 0.0419  {'class_weight': 'balanced_subsample', 'max_depth': 20, 'min_samples_leaf': 5}  (0.2 min)


  kNN (new)              best inner-CV macro-F1 0.0350  {'n_neighbors': 15, 'weights': 'uniform'}  (0.2 min)


In [6]:
import shap
rs = np.random.RandomState(RND)
dev_sites = np.unique(gdev); pick_sites = rs.choice(dev_sites, size=min(len(dev_sites), 600), replace=False)
cand = np.where(np.isin(gdev, pick_sites))[0]
sh_idx = rs.choice(cand, size=N_SHAP, replace=False); bg_idx = rs.choice(np.setdiff1d(np.arange(len(Xdev)), sh_idx), size=N_BG, replace=False)
Xsh, Xbg = Xdev.iloc[sh_idx][ORIG7], Xdev.iloc[bg_idx][ORIG7]

def shap_for(name):
    model = FINAL_B[name]["est"]
    f = lambda A: full_proba(model, pd.DataFrame(A, columns=ORIG7).astype({c: float for c in ORIG_NUM}))
    ex = shap.KernelExplainer(f, Xbg.values.astype(object))
    sv = ex.shap_values(Xsh.values.astype(object), nsamples=2 ** len(ORIG7) - 2, silent=True)
    return np.stack(sv, -1) if isinstance(sv, list) else sv      # (samples, features, classes)
def run_shap():
    out = {}
    for n in list(SPECS)[:4]:
        t0 = time.time(); out[n] = shap_for(n); print(f"  SHAP {n}: {time.time()-t0:.0f}s", flush=True)
    return out
SHAP = cached("shap", run_shap)
imp = pd.DataFrame({n: np.abs(v).mean(0).sum(1) for n, v in SHAP.items()}, index=ORIG7)
share = imp / imp.sum() * 100
rank = share.rank(ascending=False).astype(int)
display(share.round(1).assign(**{"mean_share": share.mean(1).round(1)}).sort_values("mean_share", ascending=False))
share.round(2).to_csv(f"{RES}/shap_importance_share_original7.csv"); imp.round(5).to_csv(f"{RES}/shap_importance_abs_original7.csv")
RESULTS["shap_share"] = share.round(2).to_dict()
# additivity check: SHAP values + base value reproduce the model output on the explained rows
n0 = list(SPECS)[1]; model = FINAL_B[n0]["est"]
base = full_proba(model, Xbg).mean(0); pr = full_proba(model, Xsh)
print(f"Additivity check ({n0}): max |base + sum(SHAP) - prediction| = {np.abs(base + SHAP[n0].sum(1) - pr).max():.4f} (should be ~0)")

order = share.mean(1).sort_values().index
fig, axes = plt.subplots(1, 4, figsize=(17, 4.4), sharey=True)
for ax, n in zip(axes, share.columns):
    ax.barh(order, share.loc[order, n], color=BLUE); ax.set_title(n); ax.set_xlabel("share of total mean |SHAP| (%)")
    for i, v in enumerate(share.loc[order, n]): ax.text(v + 0.4, i, f"{v:.0f}", va="center", fontsize=8)
plt.suptitle("SHAP: contribution of each original feature, by model (development sites, tuned models)", fontweight="bold", y=1.03)
plt.tight_layout(); savefig("shap_original7_by_model"); plt.show()

# per-genus view for the Random Forest: which feature moves the probability of each common genus
top_g = pd.Series(ydev).value_counts().index[:6]; v = SHAP["Random Forest"]
gm = pd.DataFrame({le.classes_[g]: np.abs(v[:, :, g]).mean(0) for g in top_g}, index=ORIG7)
fig, ax = plt.subplots(figsize=(9, 4.2)); (gm / gm.sum() * 100).T.plot.bar(stacked=True, ax=ax, colormap="tab10", width=0.75)
ax.set_ylabel("share of mean |SHAP| for that genus (%)"); ax.set_title("Random Forest: what drives the prediction of each of the 6 most common genera")
ax.legend(title="feature", bbox_to_anchor=(1.01, 1)); plt.xticks(rotation=30, ha="right"); plt.tight_layout(); savefig("shap_rf_by_genus"); plt.show()
gm.round(5).to_csv(f"{RES}/shap_rf_per_genus.csv")

# direction of the effect: Random Forest, LON_DD and lcond (all genera pooled, signed by predicted-class output)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, f_ in zip(axes, ["LON_DD", "LAT_DD", "lcond"]):
    j = ORIG7.index(f_); xv = Xsh[f_].astype(float).values; sv_ = np.abs(v[:, j, :]).sum(1)
    ax.scatter(xv, sv_, s=10, alpha=0.55, color=BLUE); ax.set_xlabel(f_); ax.set_ylabel("total |SHAP| across genera")
    ax.set_title(f"Random Forest: effect size of {f_}")
plt.tight_layout(); savefig("shap_rf_dependence"); plt.show()

  SHAP Logistic Regression: 14s


  SHAP Random Forest: 26s


  SHAP Gradient Boosting: 30s


  SHAP Neural Network: 14s


,Logistic Regression,Random Forest,Gradient Boosting,Neural Network,mean_share
LON_DD,25.2,22.2,33.4,25.8,26.7
LAT_DD,18.2,16.6,20.0,17.0,18.0
ECOREGL3,19.4,15.3,8.2,15.6,14.6
state,19.1,15.3,9.0,12.6,14.0
lcond,7.2,12.8,14.1,12.4,11.6
PHSTVL,7.0,10.4,10.3,10.8,9.6
month,4.0,7.4,5.0,5.8,5.6


Additivity check (Random Forest): max |base + sum(SHAP) - prediction| = 0.0000 (should be ~0)


**Reading the SHAP results (original seven features).** In all four models, *where the stream is* dominates. LAT_DD + LON_DD + state + ECOREGL3 together carry 82% (Logistic Regression), 69% (Random Forest), 71% (Gradient Boosting) and 71% (Neural Network) of the total SHAP importance. LON_DD is the single largest feature in every model (22-33%), then LAT_DD (17-20%). The two water-chemistry features matter more to the non-linear models (lcond + PHSTVL = 23-24% for RF, GB and NN) than to Logistic Regression (14%). `month` is last everywhere (4-7%).
The linear model leans on the categorical location codes (state and ecoregion about 19% each), while Gradient Boosting reads location mostly through the two coordinates (LON 33%) and gives ecoregion/state only 8-9%. The tree-based and neural models therefore use the same information in different ways. Per genus, the Random Forest relies most on location codes for northern cold-water genera such as *Rhinichthys* and *Semotilus* (ecoregion and state have the largest effects), while for *Lepomis* (sunfish) location codes matter less and conductivity matters relatively more. SHAP shows what the model uses, not what causes fish to occur: location is largely a stand-in for unmeasured habitat.

## 5. Feature selection inside a training set: RFE + permutation importance
`run_selection(train)` uses **only the given training rows**:
1. **RFE (RFECV-style, by original column).** A Random Forest is fit with all 27 candidate columns; the two least important columns (impurity importance summed over each column's encoded pieces)
   are removed, and this repeats. Every subset is scored with 3-fold grouped CV on macro-F1 *inside the training rows*; the best-scoring subset size is kept (ties go to the smaller subset within 1 SE).
2. **Permutation importance with noise probes.** Three random-noise columns are added. On each of 3 grouped inner splits, every column is shuffled on the inner-validation sites (3 repeats) and the drop in macro-F1 and Top-3 is recorded.
   A column passes if its drop beats the **largest probe drop** in at least 2 of the 3 splits and its mean Top-3 drop is positive.
3. A column is **proposed as an addition** only if it passes **both** tests and is not already in the original 7. The expanded set = original 7 + proposed additions (the original 7 are never removed, so arm C differs from arm B only by the additions).

In [7]:
def rf_sel(): return RandomForestClassifier(n_estimators=40 if not FAST else 20, max_depth=12, min_samples_leaf=5, class_weight="balanced_subsample", random_state=RND, n_jobs=N_JOBS)
def col_importance(pipe, feats):
    pre, rf = pipe.named_steps["pre"], pipe.named_steps["model"]
    names = pre.get_feature_names_out(); imp_ = rf.feature_importances_; out = dict.fromkeys(feats, 0.0)
    for nm, v in zip(names, imp_):
        body = nm.split("__", 1)[1]
        for f_ in sorted(feats, key=len, reverse=True):
            if body == f_ or body.startswith(f_ + "_") or body == "missingindicator_" + f_ or body.startswith("missingindicator_" + f_):
                out[f_] += v; break
    return out

def run_selection(Xtr, ytr, gtr, tag=""):
    t0 = time.time(); inner = list(StratifiedGroupKFold(3, shuffle=True, random_state=RND + 7).split(Xtr, ytr, gtr))
    # ---- RFE by column ----
    feats = list(CANDIDATES); path = []
    while True:
        sc = []; ci = dict.fromkeys(feats, 0.0)
        for tr, va in inner:
            p = Pipeline([("pre", build_pre(feats)), ("model", rf_sel())]).fit(Xtr.iloc[tr][feats], ytr[tr])
            sc.append(f1_score(ytr[va], p.predict(Xtr.iloc[va][feats]), average="macro", zero_division=0))
            for k_, v_ in col_importance(p, feats).items(): ci[k_] += v_ / len(inner)
        path.append(dict(n_features=len(feats), cv_macro_f1=np.mean(sc), cv_se=np.std(sc) / np.sqrt(len(sc)), features=list(feats)))
        if len(feats) <= 3: break
        drop = sorted(feats, key=lambda f_: ci[f_])[:2]; feats = [f_ for f_ in feats if f_ not in drop]
    P = pd.DataFrame(path); best = P.cv_macro_f1.max(); se = P.loc[P.cv_macro_f1.idxmax(), "cv_se"]
    ok = P[P.cv_macro_f1 >= best - se]; rfe_row = ok.sort_values("n_features").iloc[0]; rfe_set = list(rfe_row.features)
    # ---- permutation importance with probes ----
    rs_ = np.random.RandomState(RND); Xp = Xtr.copy(); PROBES = ["probe_1", "probe_2", "probe_3"]
    for pr_ in PROBES: Xp[pr_] = rs_.normal(size=len(Xp))
    allc = list(CANDIDATES) + PROBES; drops_f1 = {c: [] for c in allc}; drops_t3 = {c: [] for c in allc}
    global GEO_NUM, CHEM_NUM
    saved = (list(GEO_NUM), list(CHEM_NUM)); CHEM_NUM = CHEM_NUM + PROBES
    try:
        for tr, va in inner:
            p = Pipeline([("pre", build_pre(allc)), ("model", rf_sel())]).fit(Xp.iloc[tr][allc], ytr[tr])
            Xv = Xp.iloc[va][allc].reset_index(drop=True); yv = ytr[va]
            base_f, base_t = f1_score(yv, p.predict(Xv), average="macro", zero_division=0), top3_of(full_proba(p, Xv), yv)
            for c in allc:
                df_, dt_ = [], []
                for r in range(3):
                    Xs = Xv.copy(); Xs[c] = Xs[c].sample(frac=1, random_state=RND + r).values
                    pf = p.predict_proba(Xs); fp = np.zeros((len(Xs), K)); fp[:, p.classes_] = pf
                    df_.append(base_f - f1_score(yv, fp.argmax(1), average="macro", zero_division=0)); dt_.append(base_t - top3_of(fp, yv))
                drops_f1[c].append(np.mean(df_)); drops_t3[c].append(np.mean(dt_))
    finally:
        CHEM_NUM = saved[1]
    pf1 = pd.DataFrame(drops_f1).T; pt3 = pd.DataFrame(drops_t3).T
    pmax = pf1.loc[PROBES].max(0)
    perm = pd.DataFrame({"mean_drop_macro_f1": pf1.mean(1), "mean_drop_top3": pt3.mean(1), "splits_beating_probes": (pf1 > pmax.values[None, :]).sum(1)}).loc[CANDIDATES]
    perm["passes"] = (perm.splits_beating_probes >= 2) & (perm.mean_drop_top3 > 0)
    perm_set = list(perm.index[perm.passes])
    adds = [c for c in CANDIDATES if c in rfe_set and c in perm_set and c not in ORIG7]
    print(f"  selection {tag}: RFE keeps {len(rfe_set)} cols, permutation keeps {len(perm_set)}, additions = {adds}  ({(time.time()-t0)/60:.1f} min)", flush=True)
    return dict(rfe_path=P.drop(columns="features").assign(n_feat=P.n_features), rfe_set=rfe_set, perm=perm, perm_set=perm_set, additions=adds, probe_max=float(pmax.mean()))

## 6. Nested evaluation: arms A, B and C on identical outer folds
For every outer fold: select features on its training sites -> expanded set; fit arm A (fixed settings, 7 features);
GridSearchCV arm B (7 features); GridSearchCV arm C (expanded features, same grids). All three are scored on the outer-validation sites.
The loop saves a checkpoint after each fold so it can resume.

In [8]:
def nested():
    ck = f"{CACHE}/nested_checkpoint{'_fast' if FAST else ''}.pkl"
    NEST, SEL, DONE = ([], [], 0)
    if os.path.exists(ck): NEST, SEL, DONE = pickle.load(open(ck, "rb"))
    for k, (tr, va) in enumerate(OUTER):
        if k < DONE: continue
        t0 = time.time(); Xtr, ytr, gtr = Xdev.iloc[tr].reset_index(drop=True), ydev[tr], gdev[tr]; Xva, yva = Xdev.iloc[va].reset_index(drop=True), ydev[va]
        sel = run_selection(Xtr, ytr, gtr, tag=f"fold {k+1}"); SEL.append(sel); EXP = ORIG7 + sel["additions"]
        for n in SPECS:
            t1 = time.time(); m = fit_fixed(n, ORIG7, Xtr, ytr); NEST.append(dict(fold=k + 1, model=n, arm="A: original 7, fixed", features=len(ORIG7), params="fixed", inner_cv_macro_f1=np.nan, **evaluate(m, Xva[ORIG7], yva)[0]))
            for arm, fs in (("B: original 7, tuned", ORIG7), ("C: expanded, tuned", EXP)):
                gs = tune(n, fs, Xtr, ytr, gtr)
                NEST.append(dict(fold=k + 1, model=n, arm=arm, features=len(fs), params=str(best_params(gs)), inner_cv_macro_f1=gs.best_score_, **evaluate(gs.best_estimator_, Xva[fs], yva)[0]))
            print(f"    fold {k+1} {n:22s} {(time.time()-t1)/60:.1f} min", flush=True)
        pickle.dump((NEST, SEL, k + 1), open(ck, "wb")); pd.DataFrame(NEST).to_csv(f"{RES}/nested_cv_all_folds.csv", index=False)
        print(f"== outer fold {k+1}/{len(OUTER)} done in {(time.time()-t0)/60:.1f} min", flush=True)
    return NEST, SEL
NEST, SEL = nested(); N = pd.DataFrame(NEST); N.to_csv(f"{RES}/nested_cv_all_folds.csv", index=False)
print(f"Nested CV finished: {N.fold.nunique()} outer folds x {N.model.nunique()} models x 3 arms")

  selection fold 1: RFE keeps 9 cols, permutation keeps 15, additions = ['YEAR', 'Temp', 'LWSKM2', 'HUC']  (0.7 min)


    fold 1 Logistic Regression    1.0 min


    fold 1 Random Forest          0.7 min


    fold 1 Gradient Boosting      2.4 min


    fold 1 Neural Network         0.6 min


    fold 1 Extra Trees (new)      0.5 min


... (31 outputs left out)

    fold 5 Extra Trees (new)      0.4 min


    fold 5 kNN (new)              0.3 min


== outer fold 5/5 done in 6.4 min


Nested CV finished: 5 outer folds x 6 models x 3 arms


### 6a. Which features were selected, and how stable is the choice?

In [9]:
stab = pd.DataFrame({"selected_by_RFE": [sum(c in s["rfe_set"] for s in SEL) for c in CANDIDATES],
                     "passed_permutation": [sum(c in s["perm_set"] for s in SEL) for c in CANDIDATES],
                     "proposed_addition": [sum(c in s["additions"] for s in SEL) for c in CANDIDATES],
                     "mean_perm_drop_macroF1": [np.mean([s["perm"].loc[c, "mean_drop_macro_f1"] for s in SEL]) for c in CANDIDATES],
                     "mean_perm_drop_top3": [np.mean([s["perm"].loc[c, "mean_drop_top3"] for s in SEL]) for c in CANDIDATES],
                     "original7": [c in ORIG7 for c in CANDIDATES]}, index=CANDIDATES)
stab = stab.sort_values(["proposed_addition", "passed_permutation", "mean_perm_drop_macroF1"], ascending=False)
display(stab.round(4)); stab.round(5).to_csv(f"{RES}/selection_stability_by_outer_fold.csv")
print("Probe noise level (largest permutation drop among random-noise columns, avg over folds):", round(np.mean([s["probe_max"] for s in SEL]), 5))
print("RFE subset sizes chosen per outer fold:", [len(s["rfe_set"]) for s in SEL]); print("Additions per outer fold:", [s["additions"] for s in SEL])
RESULTS["stability"] = stab.round(5).reset_index().rename(columns={"index": "feature"}).to_dict(orient="records")
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for s_, i in zip(SEL, range(len(SEL))): axes[0].plot(s_["rfe_path"]["n_feat"], s_["rfe_path"]["cv_macro_f1"], marker="o", ms=3, label=f"outer fold {i+1}")
axes[0].invert_xaxis(); axes[0].set_xlabel("number of candidate columns kept"); axes[0].set_ylabel("inner CV macro-F1"); axes[0].set_title("RFE: score as columns are removed"); axes[0].legend()
st2 = stab.sort_values("mean_perm_drop_macroF1"); axes[1].barh(st2.index, st2.mean_perm_drop_macroF1, color=[ORANGE if o else BLUE for o in st2.original7])
axes[1].axvline(np.mean([s["probe_max"] for s in SEL]), color="k", ls="--", lw=1, label="largest random-probe drop"); axes[1].set_xlabel("mean drop in macro-F1 when shuffled"); axes[1].set_title("Permutation importance (orange = original 7)"); axes[1].legend()
plt.tight_layout(); savefig("selection_rfe_permutation"); plt.show()

,selected_by_RFE,passed_permutation,proposed_addition,mean_perm_drop_macroF1,mean_perm_drop_top3,original7
LWSKM2,5,5,5,0.0029,0.0071,False
HUC,4,4,4,0.0020,0.0028,False
Temp,4,3,3,0.0008,0.0017,False
YEAR,4,3,2,0.0007,0.0027,False
tn,1,1,1,-0.0001,0.0001,False
ECOREGL3,0,5,0,0.0029,0.0063,True
LAT_DD,5,5,0,0.0025,0.0106,True
lcond,5,4,0,0.0012,0.0076,True
ratio,0,4,0,0.0011,0.0004,False
hardness,0,3,0,0.0015,0.0018,False


Probe noise level (largest permutation drop among random-noise columns, avg over folds): 0.00083
RFE subset sizes chosen per outer fold: [9, 11, 11, 5, 11]
Additions per outer fold: [['YEAR', 'Temp', 'LWSKM2', 'HUC'], ['YEAR', 'Temp', 'LWSKM2', 'HUC'], ['LWSKM2', 'HUC'], ['LWSKM2'], ['Temp', 'LWSKM2', 'tn', 'HUC']]


**Reading the selection results.** Across the five outer folds, only a few extra columns survive *both* RFE and the permutation test with random-noise probes: `LWSKM2` (watershed area, added in 5 of 5 folds), `HUC` (watershed code, 4 of 5), `Temp` (stream temperature, 3 of 5) and `YEAR` (2 of 5); `tn` once. The permutation drops are small (macro-F1 drop of 0.001-0.004 for the best features) because the whole problem has weak signal, but they sit above the random-probe noise level. On all development sites the final selection proposes six additions: YEAR, Temp, LWSKM2, NO3, alk and HUC. **NO3 and alk were never selected in any outer fold**, so those two are weakly supported and should be read with caution; LWSKM2, HUC and Temp are the stable ones.

### 6b. Nested-CV results (outer-validation sites; selection and tuning never saw them)

In [10]:
M = ["macro_f1", "accuracy", "top3"]
summ = N.groupby(["model", "arm"])[M + ["inner_cv_macro_f1"]].mean().round(4); summ_sd = N.groupby(["model", "arm"])[["macro_f1"]].std().round(4).rename(columns={"macro_f1": "macro_f1_sd"})
summ = summ.join(summ_sd); display(summ); summ.to_csv(f"{RES}/nested_cv_summary.csv")
arm_mean = N.groupby("arm")[M].mean().round(4); display(arm_mean); arm_mean.to_csv(f"{RES}/nested_cv_arm_means.csv")
# paired effects across (model x fold) pairs
pv = N.pivot_table(index=["model", "fold"], columns="arm", values=M)
eff = []
for a, b, lab in [("A: original 7, fixed", "B: original 7, tuned", "tuning effect (B - A)"), ("B: original 7, tuned", "C: expanded, tuned", "feature effect (C - B)"), ("A: original 7, fixed", "C: expanded, tuned", "total (C - A)")]:
    for m_ in M:
        d = (pv[(m_, b)] - pv[(m_, a)]).values; t = stats.ttest_1samp(d, 0) if len(d) > 2 else None
        eff.append(dict(comparison=lab, metric=m_, mean_diff=d.mean(), sd=d.std(ddof=1), n_pairs=len(d), wins=int((d > 0).sum()), p_value=getattr(t, "pvalue", np.nan)))
eff = pd.DataFrame(eff); display(eff.round(4)); eff.round(5).to_csv(f"{RES}/nested_cv_paired_effects.csv", index=False)
RESULTS["nested_summary"] = summ.reset_index().to_dict(orient="records"); RESULTS["nested_arm_means"] = arm_mean.reset_index().to_dict(orient="records"); RESULTS["nested_effects"] = eff.to_dict(orient="records")
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6)); mods = list(SPECS); w = 0.27
for ax, m_ in zip(axes, M):
    for j, (arm, c) in enumerate(zip(["A: original 7, fixed", "B: original 7, tuned", "C: expanded, tuned"], [GREY, BLUE, ORANGE])):
        v_ = [N[(N.model == n) & (N.arm == arm)][m_].mean() for n in mods]; e_ = [N[(N.model == n) & (N.arm == arm)][m_].std() for n in mods]
        ax.bar(np.arange(len(mods)) + (j - 1) * w, v_, w, yerr=e_, capsize=2, color=c, label=arm)
    ax.set_xticks(range(len(mods))); ax.set_xticklabels([n.replace(" (new)", "*").replace(" ", "\n", 1) for n in mods], fontsize=8); ax.set_title({"macro_f1": "Macro-F1", "accuracy": "Accuracy", "top3": "Top-3"}[m_] + " (nested outer folds)")
axes[0].legend(fontsize=8); plt.suptitle("Three arms on identical outer folds (* = model added in this revision; bars = mean, whiskers = SD over outer folds)", fontweight="bold", y=1.03)
plt.tight_layout(); savefig("nested_cv_arms"); plt.show()

macro_f1  accuracy    top3  \
model               arm                                                
Extra Trees (new)   A: original 7, fixed    0.0426    0.0634  0.1935   
                    B: original 7, tuned    0.0426    0.0634  0.1935   
                    C: expanded, tuned      0.0468    0.0658  0.2012   
Gradient Boosting   A: original 7, fixed    0.0342    0.0508  0.1505   
                    B: original 7, tuned    0.0405    0.0585  0.1803   
                    C: expanded, tuned      0.0459    0.0621  0.1932   
Logistic Regression A: original 7, fixed    0.0252    0.0429  0.1254   
                    B: original 7, tuned    0.0252    0.0429  0.1254   
                    C: expanded, tuned      0.0339    0.0521  0.1531   
Neural Network      A: original 7, fixed    0.0270    0.0478  0.1389   
                    B: original 7, tuned    0.0284    0.0493  0.1405   
                    C: expanded, tuned      0.0321    0.0549  0.1589   
Random Forest       A: original 7, fixed    0.0458    0.0702  0.2141   
                    B: original 7, tuned    0.0458    0.0702  0.2141   
                    C: expanded, tuned      0.0523    0.0744  0.2243   
kNN (new)           A: original 7, fixed    0.0312    0.1547  0.3472   
                    B: original 7, tuned    0.0340    0.1452  0.3180   
                    C: expanded, tuned      0.0373    0.1484  0.3246   

                                          inner_cv_macro_f1  macro_f1_sd  
model               arm                                                   
Extra Trees (new)   A: original 7, fixed                NaN       0.0011  
                    B: original 7, tuned             0.0403       0.0011  
                    C: expanded, tuned               0.0441       0.0038  
Gradient Boosting   A: original 7, fixed                NaN       0.0023  
                    B: original 7, tuned             0.0402       0.0017  
                    C: expanded, tuned               0.0440       0.0027  
Logistic Regression A: original 7, fixed                NaN       0.0016  
                    B: original 7, tuned             0.0257       0.0016  
                    C: expanded, tuned               0.0337       0.0027  
Neural Network      A: original 7, fixed                NaN       0.0036  
                    B: original 7, tuned             0.0269       0.0022  
                    C: expanded, tuned               0.0331       0.0024  
Random Forest       A: original 7, fixed                NaN       0.0022  
                    B: original 7, tuned             0.0450       0.0022  
                    C: expanded, tuned               0.0489       0.0023  
kNN (new)           A: original 7, fixed                NaN       0.0022  
                    B: original 7, tuned             0.0339       0.0022  
                    C: expanded, tuned               0.0353       0.0030

,macro_f1,accuracy,top3
arm,,,
"A: original 7, fixed",0.0344,0.0716,0.1949
"B: original 7, tuned",0.0361,0.0716,0.1953
"C: expanded, tuned",0.0414,0.0763,0.2092


,comparison,metric,mean_diff,sd,n_pairs,wins,p_value
0,tuning effect (B - A),macro_f1,0.0017,0.0032,30,12,0.0057
1,tuning effect (B - A),accuracy,-0.0001,0.0056,30,9,0.9579
2,tuning effect (B - A),top3,0.0004,0.0181,30,7,0.9114
3,feature effect (C - B),macro_f1,0.0053,0.0034,30,29,0.0000
4,feature effect (C - B),accuracy,0.0047,0.0043,30,25,0.0000
5,feature effect (C - B),top3,0.0139,0.0108,30,30,0.0000
6,total (C - A),macro_f1,0.0070,0.0041,30,28,0.0000
7,total (C - A),accuracy,0.0046,0.0064,30,24,0.0004
8,total (C - A),top3,0.0143,0.0214,30,25,0.0010


**Reading the nested-CV results.** Averaged over the 6 models and 5 outer folds: Arm A (7 features, fixed) macro-F1 0.0344 / accuracy 0.0716 / Top-3 0.1949; Arm B (7 features, tuned) 0.0361 / 0.0716 / 0.1953; Arm C (expanded, tuned) 0.0414 / 0.0763 / 0.2092. **Tuning on its own moved almost nothing** (+0.0017 macro-F1, +0.0004 Top-3): for Logistic Regression, Random Forest and Extra Trees the grid picked the same settings as the fixed defaults. **The expanded features helped consistently**: C beat B in 29 of 30 model-by-fold pairs on macro-F1 and 30 of 30 on Top-3 (about +0.005 macro-F1 and +1.4 points Top-3). The paired p-values treat the 30 pairs as independent, which they are not (folds share data and models see the same rows), so the win counts and the size of the effect are more trustworthy than the p-values.

## 7. Final selection and final tuning on the whole development set
For the final models, the selection procedure is run once more on **all development sites** (still no test sites). The tuned **arm B** models
were already fit above (Section 4). Here the expanded feature list is fixed and arm C is tuned with the identical grids.

In [11]:
FINAL_SEL = cached("final_sel", lambda: run_selection(Xdev, ydev, gdev, tag="final (all development sites)"))
ADDITIONS = FINAL_SEL["additions"]; EXPANDED = ORIG7 + ADDITIONS
print("Original 7:", ORIG7); print("Final additions:", ADDITIONS if ADDITIONS else "none"); print("Expanded set:", EXPANDED)
fs_tbl = pd.DataFrame({"RFE_selected": [c in FINAL_SEL["rfe_set"] for c in CANDIDATES], "perm_drop_macro_f1": FINAL_SEL["perm"].mean_drop_macro_f1.values, "perm_drop_top3": FINAL_SEL["perm"].mean_drop_top3.values,
                       "splits_beating_probes": FINAL_SEL["perm"].splits_beating_probes.values, "perm_passes": FINAL_SEL["perm"].passes.values, "original7": [c in ORIG7 for c in CANDIDATES],
                       "proposed_addition": [c in ADDITIONS for c in CANDIDATES]}, index=CANDIDATES).sort_values("perm_drop_macro_f1", ascending=False)
display(fs_tbl.round(4)); fs_tbl.round(5).to_csv(f"{RES}/final_feature_selection.csv")
RESULTS["final_selection"] = dict(additions=ADDITIONS, rfe_set=FINAL_SEL["rfe_set"], perm_set=FINAL_SEL["perm_set"])

def tune_C_final():
    out = {}
    for n in SPECS:
        t0 = time.time(); gs = tune(n, EXPANDED, Xdev, ydev, gdev)
        out[n] = dict(est=gs.best_estimator_, params=best_params(gs), cv=pd.DataFrame(gs.cv_results_), best_cv=gs.best_score_, secs=time.time() - t0)
        print(f"  {n:22s} best inner-CV macro-F1 {gs.best_score_:.4f}  {best_params(gs)}  ({out[n]['secs']/60:.1f} min)", flush=True)
    return out
FINAL_C = cached("final_C", tune_C_final) if ADDITIONS else None

  selection final (all development sites): RFE keeps 13 cols, permutation keeps 12, additions = ['YEAR', 'Temp', 'LWSKM2', 'NO3', 'alk', 'HUC']  (0.8 min)


Original 7: ['LAT_DD', 'LON_DD', 'month', 'lcond', 'PHSTVL', 'state', 'ECOREGL3']
Final additions: ['YEAR', 'Temp', 'LWSKM2', 'NO3', 'alk', 'HUC']
Expanded set: ['LAT_DD', 'LON_DD', 'month', 'lcond', 'PHSTVL', 'state', 'ECOREGL3', 'YEAR', 'Temp', 'LWSKM2', 'NO3', 'alk', 'HUC']


,RFE_selected,perm_drop_macro_f1,perm_drop_top3,splits_beating_probes,perm_passes,original7,proposed_addition
HUC,True,0.0042,0.0013,3,True,False,True
LAT_DD,True,0.0031,0.0072,3,True,True,False
lcond,True,0.0027,0.0071,3,True,True,False
LWSKM2,True,0.0027,0.0061,3,True,False,True
ECOREGL3,False,0.0023,0.0042,3,True,True,False
LON_DD,True,0.0022,0.0034,3,True,True,False
YEAR,True,0.0015,0.0018,2,True,False,True
Temp,True,0.0014,0.0006,3,True,False,True
hardness,False,0.0014,0.0015,2,True,False,False
state,False,0.0012,0.0033,2,True,True,False


  Logistic Regression    best inner-CV macro-F1 0.0352  {'C': 0.1, 'class_weight': 'balanced'}  (1.0 min)


  Random Forest          best inner-CV macro-F1 0.0523  {'class_weight': 'balanced_subsample', 'max_depth': 20, 'min_samples_leaf': 5}  (0.4 min)


  Gradient Boosting      best inner-CV macro-F1 0.0453  {'class_weight': 'balanced', 'learning_rate': 0.15, 'max_depth': 5}  (1.8 min)


  Neural Network         best inner-CV macro-F1 0.0381  {'alpha': 0.01, 'hidden_layer_sizes': [64]}  (0.4 min)


  Extra Trees (new)      best inner-CV macro-F1 0.0463  {'class_weight': 'balanced_subsample', 'max_depth': 20, 'min_samples_leaf': 5}  (0.3 min)


  kNN (new)              best inner-CV macro-F1 0.0361  {'n_neighbors': 15, 'weights': 'uniform'}  (0.3 min)


## 8. Held-out test sites: final evaluation (used once)
**Do not confuse the two kinds of numbers.** *Best grid-search score* = mean inner-CV macro-F1 on development sites, the number GridSearchCV maximised (optimistically selected).
*Test result* = the model refit on all development sites and scored once on test sites that were never used for selection or tuning.

In [12]:
def fit_A_final(n): return fit_fixed(n, ORIG7, Xdev, ydev)
FINAL_A = cached("final_A", lambda: {n: fit_A_final(n) for n in SPECS})
arms = {"A: original 7, fixed": (ORIG7, {n: FINAL_A[n] for n in SPECS}), "B: original 7, tuned": (ORIG7, {n: FINAL_B[n]["est"] for n in SPECS})}
if FINAL_C: arms["C: expanded, tuned"] = (EXPANDED, {n: FINAL_C[n]["est"] for n in SPECS})
TEST, PRED, FULL = [], {}, {}
for arm, (fs, ms) in arms.items():
    for n, m in ms.items():
        d, pred, full = evaluate(m, Xte[fs], yte, extra=True); PRED[(arm, n)] = pred; FULL[(arm, n)] = full
        cvb = {"A: original 7, fixed": np.nan, "B: original 7, tuned": FINAL_B[n]["best_cv"], "C: expanded, tuned": (FINAL_C[n]["best_cv"] if FINAL_C else np.nan)}[arm]
        TEST.append(dict(model=n, arm=arm, n_features=len(fs), best_gridsearch_cv_macro_f1=cvb, test_macro_f1=d["macro_f1"], test_accuracy=d["accuracy"], test_top3=d["top3"], test_roc_auc=d["roc_auc"], test_log_loss=d["log_loss"]))
T = pd.DataFrame(TEST)
base_pred = np.full(len(yte), np.bincount(ydev).argmax())
base = dict(model="Baseline: always most common genus", arm="-", n_features=0, test_macro_f1=f1_score(yte, base_pred, average="macro", zero_division=0), test_accuracy=accuracy_score(yte, base_pred), test_top3=np.nan)
T = pd.concat([T, pd.DataFrame([base])], ignore_index=True); T.round(5).to_csv(f"{RES}/final_test_results.csv", index=False)
display(T.round(4)); RESULTS["test"] = T.round(5).to_dict(orient="records")
# hyperparameters actually selected (final fits)
hp = pd.DataFrame([dict(model=n, arm=a, selected_hyperparameters=str((FINAL_B if a.startswith("B") else FINAL_C)[n]["params"]), best_gridsearch_cv_macro_f1=(FINAL_B if a.startswith("B") else FINAL_C)[n]["best_cv"],
        tuning_minutes=(FINAL_B if a.startswith("B") else FINAL_C)[n]["secs"] / 60) for a in arms if not a.startswith("A") for n in SPECS])
hp.round(5).to_csv(f"{RES}/selected_hyperparameters.csv", index=False); display(hp.round(4)); RESULTS["hyper"] = hp.round(5).to_dict(orient="records")
for a in arms:
    if a.startswith("A"): continue
    store = FINAL_B if a.startswith("B") else FINAL_C
    for n in SPECS: store[n]["cv"].assign(params_str=store[n]["cv"]["params"].astype(str).str.replace("model__", "")).drop(columns=["params"]).to_csv(f"{RES}/gridsearch_cv_{a[0]}_{n.split()[0].lower()}.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, m_ in zip(axes, ["test_macro_f1", "test_accuracy", "test_top3"]):
    for j, (arm, c) in enumerate(zip(arms, [GREY, BLUE, ORANGE])):
        v_ = [T[(T.model == n) & (T.arm == arm)][m_].iloc[0] for n in SPECS]; ax.bar(np.arange(len(SPECS)) + (j - 1) * 0.27, v_, 0.27, color=c, label=arm)
    ax.axhline(T[T.model.str.startswith("Baseline")][m_].iloc[0] if m_ != "test_top3" else 0, color="k", ls=":", lw=1)
    ax.set_xticks(range(len(SPECS))); ax.set_xticklabels([n.replace(" (new)", "*").replace(" ", "\n", 1) for n in SPECS], fontsize=8); ax.set_title(m_.replace("test_", "Test ").replace("_", "-"))
axes[0].legend(fontsize=8); plt.suptitle("Held-out test sites, final models (dotted = always-most-common-genus baseline)", fontweight="bold", y=1.03); plt.tight_layout(); savefig("final_test_arms"); plt.show()

# grid-search vs test comparison (optimism of the tuned score)
gap = T[T.arm.str.startswith(("B", "C"))][["model", "arm", "best_gridsearch_cv_macro_f1", "test_macro_f1"]].copy(); gap["optimism"] = gap.best_gridsearch_cv_macro_f1 - gap.test_macro_f1
display(gap.round(4)); gap.round(5).to_csv(f"{RES}/gridsearch_vs_test.csv", index=False)

,model,arm,n_features,best_gridsearch_cv_macro_f1,test_macro_f1,test_accuracy,test_top3,test_roc_auc,test_log_loss
0,Logistic Regression,"A: original 7, fixed",7,NaN,0.0282,0.0443,0.1296,0.7572,3.5375
1,Random Forest,"A: original 7, fixed",7,NaN,0.0491,0.0712,0.2177,0.7986,3.1800
2,Gradient Boosting,"A: original 7, fixed",7,NaN,0.0344,0.0508,0.1565,0.7796,3.4468
3,Neural Network,"A: original 7, fixed",7,NaN,0.0311,0.0520,0.1455,0.7675,3.4899
4,Extra Trees (new),"A: original 7, fixed",7,NaN,0.0465,0.0644,0.1916,0.7990,3.2415
5,kNN (new),"A: original 7, fixed",7,NaN,0.0307,0.1556,0.3447,0.6676,4.5778
6,Logistic Regression,"B: original 7, tuned",7,0.0252,0.0282,0.0443,0.1296,0.7572,3.5375
7,Random Forest,"B: original 7, tuned",7,0.0467,0.0491,0.0712,0.2177,0.7986,3.1800
8,Gradient Boosting,"B: original 7, tuned",7,0.0408,0.0443,0.0579,0.1824,0.7849,3.3541
9,Neural Network,"B: original 7, tuned",7,0.0280,0.0273,0.0507,0.1362,0.7600,3.5290


,model,arm,selected_hyperparameters,best_gridsearch_cv_macro_f1,tuning_minutes
0,Logistic Regression,"B: original 7, tuned","{'C': 1.0, 'class_weight': 'balanced'}",0.0252,0.5827
1,Random Forest,"B: original 7, tuned","{'class_weight': 'balanced_subsample', 'max_de...",0.0467,0.4397
2,Gradient Boosting,"B: original 7, tuned","{'class_weight': 'balanced', 'learning_rate': ...",0.0408,1.0396
3,Neural Network,"B: original 7, tuned","{'alpha': 0.01, 'hidden_layer_sizes': [64, 32]}",0.0280,0.3686
4,Extra Trees (new),"B: original 7, tuned","{'class_weight': 'balanced_subsample', 'max_de...",0.0419,0.2181
5,kNN (new),"B: original 7, tuned","{'n_neighbors': 15, 'weights': 'uniform'}",0.0350,0.2288
6,Logistic Regression,"C: expanded, tuned","{'C': 0.1, 'class_weight': 'balanced'}",0.0352,1.0058
7,Random Forest,"C: expanded, tuned","{'class_weight': 'balanced_subsample', 'max_de...",0.0523,0.4412
8,Gradient Boosting,"C: expanded, tuned","{'class_weight': 'balanced', 'learning_rate': ...",0.0453,1.8083
9,Neural Network,"C: expanded, tuned","{'alpha': 0.01, 'hidden_layer_sizes': [64]}",0.0381,0.3679


,model,arm,best_gridsearch_cv_macro_f1,test_macro_f1,optimism
6,Logistic Regression,"B: original 7, tuned",0.0252,0.0282,-0.0030
7,Random Forest,"B: original 7, tuned",0.0467,0.0491,-0.0023
8,Gradient Boosting,"B: original 7, tuned",0.0408,0.0443,-0.0034
9,Neural Network,"B: original 7, tuned",0.0280,0.0273,0.0007
10,Extra Trees (new),"B: original 7, tuned",0.0419,0.0465,-0.0046
11,kNN (new),"B: original 7, tuned",0.0350,0.0358,-0.0008
12,Logistic Regression,"C: expanded, tuned",0.0352,0.0359,-0.0006
13,Random Forest,"C: expanded, tuned",0.0523,0.0522,0.0001
14,Gradient Boosting,"C: expanded, tuned",0.0453,0.0496,-0.0043
15,Neural Network,"C: expanded, tuned",0.0381,0.0389,-0.0008


**Reading the test results.** These come from models refit on all development sites and scored once on 656 held-out test sites. Best model: Random Forest on the expanded set (macro-F1 0.0522, accuracy 0.0751, Top-3 0.2312, ROC-AUC 0.817) versus the original-seven RF (0.0491, 0.0712, 0.2177, 0.799). The site-level bootstrap says the macro-F1 gain from added features is clear for Logistic Regression (+0.0074), Gradient Boosting (+0.0047) and the Neural Network (+0.0106), but **not** for the Random Forest (+0.0026, interval includes 0), Extra Trees or kNN. Top-3 improves with a clear interval for five of six models. Best grid-search CV scores and test scores agree within about 0.005 (differences go both ways), so tuning did not overfit the development data. **kNN looks best on accuracy (0.15) and Top-3 (0.33)** only because it mostly predicts the most common genera: its macro-F1 and ROC-AUC (0.63-0.65) are the worst, and its log-loss is highest. For comparison, the baseline that always says the most common genus has accuracy 0.125 but macro-F1 0.004, which is why accuracy alone is misleading here.

### 8a. Site-level bootstrap: is the test difference real?
Test rows from one site are not independent, so the bootstrap resamples **test sites** (not rows). It gives a 95% interval for the difference in macro-F1, accuracy and Top-3 between arms.

In [13]:
rs = np.random.RandomState(RND); sites = np.unique(gte); rows_by_site = {s_: np.where(gte == s_)[0] for s_ in sites}; NB = 200 if not FAST else 20
def boot_diff(a_pred, b_pred, a_full, b_full):
    out = []
    for _ in range(NB):
        idx = np.concatenate([rows_by_site[s_] for s_ in rs.choice(sites, len(sites))]); yy = yte[idx]
        out.append([f1_score(yy, b_pred[idx], average="macro", zero_division=0) - f1_score(yy, a_pred[idx], average="macro", zero_division=0),
                    accuracy_score(yy, b_pred[idx]) - accuracy_score(yy, a_pred[idx]), top3_of(b_full[idx], yy) - top3_of(a_full[idx], yy)])
    o = np.array(out); return o.mean(0), np.percentile(o, 2.5, 0), np.percentile(o, 97.5, 0)
bd = []
pairs = [("A: original 7, fixed", "B: original 7, tuned", "tuning (B - A)")] + ([("B: original 7, tuned", "C: expanded, tuned", "added features (C - B)")] if FINAL_C else [])
for n in SPECS:
    for a, b, lab in pairs:
        m, lo, hi = boot_diff(PRED[(a, n)], PRED[(b, n)], FULL[(a, n)], FULL[(b, n)])
        for i, mt in enumerate(["macro_f1", "accuracy", "top3"]): bd.append(dict(model=n, comparison=lab, metric=mt, diff=m[i], ci_low=lo[i], ci_high=hi[i], excludes_zero=bool(lo[i] > 0 or hi[i] < 0)))
bd = pd.DataFrame(bd); bd.round(5).to_csv(f"{RES}/test_bootstrap_differences.csv", index=False)
display(bd[bd.metric == "macro_f1"].round(4)); RESULTS["boot"] = bd.round(5).to_dict(orient="records")

,model,comparison,metric,diff,ci_low,ci_high,excludes_zero
0,Logistic Regression,tuning (B - A),macro_f1,0.0000,0.0000,0.0000,False
3,Logistic Regression,added features (C - B),macro_f1,0.0074,0.0031,0.0114,True
6,Random Forest,tuning (B - A),macro_f1,0.0000,0.0000,0.0000,False
9,Random Forest,added features (C - B),macro_f1,0.0027,-0.0018,0.0072,False
12,Gradient Boosting,tuning (B - A),macro_f1,0.0099,0.0064,0.0139,True
15,Gradient Boosting,added features (C - B),macro_f1,0.0047,0.0002,0.0092,True
18,Neural Network,tuning (B - A),macro_f1,-0.0035,-0.0070,0.0006,False
21,Neural Network,added features (C - B),macro_f1,0.0106,0.0067,0.0149,True
24,Extra Trees (new),tuning (B - A),macro_f1,0.0000,0.0000,0.0000,False
27,Extra Trees (new),added features (C - B),macro_f1,-0.0007,-0.0050,0.0032,False


## 9. Why each proposed addition? Evidence behind the justification
For each addition: how large its permutation drop is relative to noise, how often it was selected across outer folds, and what a **drop-one check** shows
(refit the best-C Random Forest without that single feature on the development sites and score on outer validation... here simplified: on the nested folds using the tuned RF settings).
This uses development data only.

In [14]:
if ADDITIONS:
    best_rf = {k.replace("model__", ""): v for k, v in FINAL_C["Random Forest"]["params"].items()} if False else FINAL_C["Random Forest"]["params"]
    dro = []
    for tr, va in OUTER:
        Xtr, ytr = Xdev.iloc[tr].reset_index(drop=True), ydev[tr]; Xva, yva = Xdev.iloc[va].reset_index(drop=True), ydev[va]
        for lab, fs in [("all expanded", EXPANDED)] + [(f"without {a}", [c for c in EXPANDED if c != a]) for a in ADDITIONS] + [("original 7 only", ORIG7)]:
            est = RandomForestClassifier(n_estimators=100, random_state=RND, n_jobs=N_JOBS, **{k: v for k, v in best_rf.items()})
            p = Pipeline([("pre", build_pre(fs)), ("model", est)]).fit(Xtr[fs], ytr); dro.append(dict(variant=lab, **evaluate(p, Xva[fs], yva)[0]))
    dro = pd.DataFrame(dro).groupby("variant").mean().round(4).sort_values("macro_f1", ascending=False); display(dro); dro.to_csv(f"{RES}/drop_one_check.csv")
    RESULTS["drop_one"] = dro.reset_index().to_dict(orient="records")
    # SHAP share of additions inside the expanded Random Forest is not computed with Kernel SHAP (too many features); TreeExplainer is exact for forests
    rfC = FINAL_C["Random Forest"]["est"]; pre = rfC.named_steps["pre"]; rows_ = Xdev.iloc[sh_idx][EXPANDED]; Z = pre.transform(rows_)
    tsh = shap.TreeExplainer(rfC.named_steps["model"]).shap_values(Z, check_additivity=False)
    tsh = np.stack(tsh, -1) if isinstance(tsh, list) else tsh
    nm = list(pre.get_feature_names_out()); per_col = dict.fromkeys(EXPANDED, 0.0); a_ = np.abs(tsh).mean(0).sum(1)
    for nme, v_ in zip(nm, a_):
        body = nme.split("__", 1)[1]
        for f_ in sorted(EXPANDED, key=len, reverse=True):
            if body == f_ or body.startswith(f_ + "_") or body.startswith("missingindicator_" + f_): per_col[f_] += v_; break
    ts = pd.Series(per_col); ts = (ts / ts.sum() * 100).sort_values(ascending=False); display(ts.round(1).to_frame("RF_expanded_SHAP_share_%")); ts.round(3).to_csv(f"{RES}/shap_share_expanded_rf.csv")
    RESULTS["shap_share_expanded"] = ts.round(3).to_dict()
else:
    print("No additions were proposed, so there is nothing to justify. Arm C is identical to arm B.")

,macro_f1,accuracy,top3
variant,,,
all expanded,0.0534,0.0748,0.2257
without HUC,0.0525,0.0766,0.2257
without alk,0.0522,0.0750,0.2239
without NO3,0.0520,0.0739,0.2243
without YEAR,0.0519,0.0742,0.2243
without Temp,0.0513,0.0725,0.2223
without LWSKM2,0.0489,0.0713,0.2175
original 7 only,0.0458,0.0702,0.2141


,RF_expanded_SHAP_share_%
LWSKM2,14.6
HUC,14.2
LON_DD,10.3
ECOREGL3,9.6
state,9.2
LAT_DD,8.1
NO3,6.3
lcond,6.1
Temp,5.3
PHSTVL,5.0


**Why each addition is useful (evidence on development data).** Removing `LWSKM2` from the expanded Random Forest hurts most (macro-F1 0.0534 to 0.0489, Top-3 0.2257 to 0.2175): watershed size separates small headwater streams (darters, dace) from larger streams (sunfish, suckers). `Temp`: removing it lowers macro-F1 to 0.0513; water temperature is the strongest cold-water versus warm-water signal and was top-ranked in the literature (Kim et al. 2021). `YEAR` (0.0519), `NO3` (0.0520) and `alk` (0.0522) each add about 0.001, consistent with the weak support seen in selection (they track sampling era and nutrient or buffering conditions). `HUC`: removing it leaves macro-F1 slightly higher (0.0525) and Top-3 unchanged, so its value is mixed even though permutation importance flagged it; it is a fine-grained location code, and fine location is what SHAP already shows the models use.

## 10. Save models, data and key results

In [15]:
rows_ = []
def save(obj, path): pickle.dump(obj, open(path, "wb")); return round(os.path.getsize(path) / 1e6, 1)
for n in SPECS:
    f = n.split()[0].lower().replace("(new)", "")
    rows_.append(dict(file=f"tuned_original7_{f}.pkl", MB=save(FINAL_B[n]["est"], f"{MOD}/tuned_original7_{f}.pkl")))
    if FINAL_C: rows_.append(dict(file=f"tuned_expanded_{f}.pkl", MB=save(FINAL_C[n]["est"], f"{MOD}/tuned_expanded_{f}.pkl")))
pickle.dump(le, open(f"{MOD}/label_encoder.pkl", "wb")); json.dump(dict(original7=ORIG7, expanded=EXPANDED), open(f"{MOD}/feature_lists.json", "w"), indent=1)
sz = pd.DataFrame(rows_); display(sz); sz.to_csv(f"{RES}/saved_models.csv", index=False)
# reload check: a saved pipeline must reproduce its predictions
ref = FINAL_C["Random Forest"]["est"] if FINAL_C else FINAL_B["Random Forest"]["est"]
chk = pickle.load(open(f"{MOD}/{'tuned_expanded_random.pkl' if FINAL_C else 'tuned_original7_random.pkl'}", "rb"))
assert np.array_equal(chk.predict(Xte[(EXPANDED if FINAL_C else ORIG7)]), ref.predict(Xte[(EXPANDED if FINAL_C else ORIG7)])), "reloaded model differs"
print("reloaded model reproduces predictions: True")
data[["ROW_ID", "SITE_ID", "GENUS"] + CANDIDATES].to_csv("cleaned_dataset_expanded.csv", index=False)
RESULTS.update(original7=ORIG7, expanded=EXPANDED, additions=ADDITIONS, n_outer=len(OUTER))
json.dump(RESULTS, open(f"{RES}/key_results_revised.json", "w"), indent=1, default=jsonable)
print("saved")

,file,MB
0,tuned_original7_logistic.pkl,0.0
1,tuned_expanded_logistic.pkl,0.0
2,tuned_original7_random.pkl,177.6
3,tuned_expanded_random.pkl,170.2
4,tuned_original7_gradient.pkl,3.3
5,tuned_expanded_gradient.pkl,3.5
6,tuned_original7_neural.pkl,0.1
7,tuned_expanded_neural.pkl,0.2
8,tuned_original7_extra.pkl,168.8
9,tuned_expanded_extra.pkl,167.4


reloaded model reproduces predictions: True


saved


## Conclusions
* **SHAP:** location (coordinates, state, ecoregion) drives every existing model (69-82% of SHAP importance); chemistry matters more for the non-linear models; month matters least.
* **Feature selection:** RFE plus permutation importance (with noise probes) inside each training fold picks LWSKM2, HUC, Temp and YEAR most often; NO3 and alk enter only in the final all-development selection and are weakly supported.
* **Tuning:** GridSearchCV with grouped 5-fold CV changed little (+0.0017 macro-F1 on average); for three models it chose the default settings.
* **Expanded vs original features:** a small, consistent improvement (about +0.005 macro-F1 and +1.4 points Top-3 in nested CV; the best test model reaches Top-3 0.231). It is statistically clear for several models, but practically small: macro-F1 stays near 0.05.
* **Limitations:** about 9 genera share each site's measurements, which caps single-genus accuracy; YEAR and HUC are not physical drivers; one held-out test split; small grids; SHAP shows model use, not causation; high missingness in several added chemistry columns (median-imputed, with missing flags).